# Phân tích Yêu cầu Chi phí

Tập ghi chú này trình bày cách tạo các đại lý sử dụng plugin để xử lý chi phí đi lại từ hình ảnh biên lai địa phương, tạo email yêu cầu chi phí và trực quan hóa dữ liệu chi phí bằng biểu đồ hình tròn. Các đại lý chọn chức năng một cách động dựa trên ngữ cảnh nhiệm vụ.

Các bước:
1. Đại lý OCR xử lý hình ảnh biên lai địa phương và trích xuất dữ liệu chi phí đi lại.
2. Đại lý Email tạo email yêu cầu chi phí.

### Ví dụ về kịch bản chi phí đi lại:
Giả sử bạn là một nhân viên đi công tác họp ở thành phố khác. Công ty bạn có chính sách hoàn trả tất cả các chi phí liên quan đến đi lại hợp lý. Dưới đây là phân tích các chi phí có thể phát sinh:
- Vận chuyển:
Vé máy bay khứ hồi từ thành phố nhà bạn đến thành phố đích.
Taxi hoặc dịch vụ gọi xe đi đến và về từ sân bay.
Vận chuyển địa phương tại thành phố đích (như giao thông công cộng, thuê xe, hoặc taxi).

- Lưu trú:
Ở khách sạn ba đêm tại khách sạn kinh doanh tầm trung gần địa điểm họp.

- Bữa ăn:
Trợ cấp bữa ăn hàng ngày cho bữa sáng, trưa và tối, dựa trên chính sách tiền ăn của công ty.

- Chi phí khác:
Phí gửi xe tại sân bay.
Chi phí truy cập internet tại khách sạn.
Tiền boa hoặc các khoản phí dịch vụ nhỏ.

- Hồ sơ:
Bạn nộp tất cả biên lai (vé máy bay, taxi, khách sạn, bữa ăn, v.v.) và báo cáo chi phí đã hoàn thành để được hoàn tiền.


## Nhập các thư viện cần thiết

Nhập các thư viện và mô-đun cần thiết cho sổ ghi chép.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Định nghĩa Mô hình Chi phí

 Tạo một mô hình Pydantic cho các khoản chi phí cá nhân và một lớp ExpenseFormatter để chuyển đổi truy vấn của người dùng thành dữ liệu chi phí có cấu trúc.

 Mỗi khoản chi phí sẽ được biểu diễn dưới định dạng:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Định nghĩa Công cụ - Tạo Email

Tạo một hàm công cụ để tạo email gửi yêu cầu bồi hoàn chi phí.
- Công cụ này sử dụng `@tool` decorator từ Microsoft Agent Framework.
- Nó tính tổng số tiền các chi phí và định dạng chi tiết thành nội dung email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Công cụ Trích xuất Chi phí Du lịch từ Ảnh Hóa đơn

Tạo một hàm công cụ để trích xuất chi phí du lịch từ ảnh hóa đơn.
- Công cụ này sử dụng `@tool` decorator từ Microsoft Agent Framework.
- Nó đọc ảnh hóa đơn, mã hóa nó dưới dạng base64 và trả về URI dữ liệu để đại lý phân tích.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Xử Lý Chi Phí

Xác định các đại lý và kết nối chúng vào một quy trình tuần tự bằng cách sử dụng `WorkflowBuilder`.
- Đại lý OCR trích xuất dữ liệu chi phí có cấu trúc từ hình ảnh hóa đơn bằng công cụ `load_receipt_image`.
- Đại lý Email lấy dữ liệu đã trích xuất và tạo email yêu cầu chi phí chuyên nghiệp bằng công cụ `generate_expense_email`.
- `WorkflowBuilder` với `add_edge` tạo một chuỗi công việc tuần tự: Đại lý OCR → Đại lý Email.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Hàm chính

Xây dựng quy trình làm việc tuần tự và chạy nó để xử lý hình ảnh hóa đơn và tạo email yêu cầu chi phí.


> **Lưu ý:** Quy trình làm việc này hiện đang truyền hình ảnh hóa đơn dưới dạng văn bản base64, mà hầu hết các mô hình trò chuyện (bao gồm cả gpt-5-mini) sẽ không xem như một hình ảnh.
> Nó cũng có thể vượt quá giới hạn cửa sổ ngữ cảnh của mô hình. Ưu tiên chạy OCR với Azure AI Vision (hoặc công cụ OCR khác) và chỉ truyền văn bản đã trích xuất, hoặc cải tiến lại để gửi hình ảnh dưới dạng tin nhắn `image_url`.
> Nếu bạn chỉ muốn tránh lỗi ngữ cảnh, hãy thử sử dụng hình ảnh hóa đơn nhỏ hơn hoặc mô hình có cửa sổ ngữ cảnh lớn hơn.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
